In [1]:
import pandas as pd
import re
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
fake_df = pd.read_csv("../data/raw/Fake.csv")
real_df = pd.read_csv("../data/raw/True.csv")

fake_df["label"] = 0  # 0 = fake
real_df["label"] = 1  # 1 = real

df = pd.concat([fake_df, real_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(df.shape)
df.head()

(44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [3]:
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_content"] = df["content"].apply(clean_text)
df[["content", "clean_content", "label"]].head()

,content,clean_content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,ben stein calls out th circuit court committed...,0
1,Trump drops Steve Bannon from National Securit...,trump drops steve bannon from national securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,puerto rico expects us to lift jones act shipp...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,oops trump just accidentally confirmed he leak...,0
4,Donald Trump heads for Scotland to reopen a go...,donald trump heads for scotland to reopen a go...,1


In [4]:
X = df["clean_content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (35918, 10000)
Test shape: (8980, 10000)


In [5]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

print("Naive Bayes Results")
print(classification_report(y_test, nb_preds, target_names=["fake", "real"]))

Naive Bayes Results
              precision    recall  f1-score   support

        fake       0.95      0.95      0.95      4696
        real       0.94      0.94      0.94      4284

    accuracy                           0.94      8980
   macro avg       0.94      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980



In [6]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print("Logistic Regression Results")
print(classification_report(y_test, lr_preds, target_names=["fake", "real"]))

Logistic Regression Results
              precision    recall  f1-score   support

        fake       0.99      0.99      0.99      4696
        real       0.99      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [7]:
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb_model.fit(X_train_tfidf, y_train)
xgb_preds = xgb_model.predict(X_test_tfidf)

print("XGBoost Results")
print(classification_report(y_test, xgb_preds, target_names=["fake", "real"]))

c:\Users\HP\Desktop\social-media-intelligence\smienv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Results
              precision    recall  f1-score   support

        fake       1.00      1.00      1.00      4696
        real       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [8]:
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, nb_preds),
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, xgb_preds)
    ],
    "F1 Score": [
        f1_score(y_test, nb_preds),
        f1_score(y_test, lr_preds),
        f1_score(y_test, xgb_preds)
    ]
})

results.sort_values("F1 Score", ascending=False)

,Model,Accuracy,F1 Score
2,XGBoost,0.998330,0.998250
1,Logistic Regression,0.989532,0.989062
0,Naive Bayes,0.944989,0.942303


In [9]:
joblib.dump(xgb_model, "../trained_models/fake_news_classifier.pkl")
joblib.dump(vectorizer, "../trained_models/fake_news_vectorizer.pkl")

print("Model and vectorizer saved successfully.")

Model and vectorizer saved successfully.


In [10]:
loaded_model = joblib.load("../trained_models/fake_news_classifier.pkl")
loaded_vectorizer = joblib.load("../trained_models/fake_news_vectorizer.pkl")

sample_text = ["Scientists confirm the earth is flat, NASA admits cover-up"]
sample_clean = [clean_text(t) for t in sample_text]
sample_tfidf = loaded_vectorizer.transform(sample_clean)

prediction = loaded_model.predict(sample_tfidf)
print("Prediction (0=fake, 1=real):", prediction)

Prediction (0=fake, 1=real): [0]
